# 01 - Plain text NER

Run GLiNER over `data/plain/validation.jsonl`, save predictions, and evaluate them with strict matching.

In [1]:
%load_ext autoreload
%autoreload 2

In [4]:
from ner.evaluation import evaluate_ner_strict
from ner.inference import load_gliner_model, predict_dataset
from ner.utils import load_jsonl, save_responses_to_jsonl
from pathlib import Path
from ner.inference import get_dataset_labels


In [3]:
VALIDATION_PATH = '../data/plain/validation.jsonl'
OUTPUT_PATH = '../results/plain/gliner_small_validation.jsonl'

LABELS = [
    'PERSON_NAME', 'ID_CARD_NUMBER', 'ADDRESS',
    'REGISTRATION_NUMBER', 'DATE', 'AMOUNT', 'ORGANIZATION'
]


In [1]:
samples = load_jsonl(VALIDATION_PATH)
model = load_gliner_model(device='cuda')
responses = predict_dataset(model, samples, LABELS, threshold=0.5)
save_responses_to_jsonl(responses, OUTPUT_PATH)


NameError: name 'load_jsonl' is not defined

In [5]:
strict_results = evaluate_ner_strict(VALIDATION_PATH, OUTPUT_PATH)
strict_results['micro']



STRICT EVALUATION
Micro -> P: 0.76 | R: 0.86 | F1: 0.81 | TP: 1773 | FP: 545 | FN: 277

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------------------------
PERSON_NAME                 1.00   1.00   1.00     500       0       0
ID_CARD_NUMBER              0.92   1.00   0.96     200      17       0
ADDRESS                     0.00   0.01   0.00       1     415     199
REGISTRATION_NUMBER         1.00   0.94   0.97     236       1      14
DATE                        1.00   1.00   1.00     399       0       1
AMOUNT                      0.95   0.92   0.94     184       9      16
ORGANIZATION                0.71   0.84   0.77     253     103      47


{'precision': 0.7648835202761001,
 'recall': 0.8648780487804878,
 'f1': 0.8118131868131867,
 'true_positives': 1773,
 'false_positives': 545,
 'false_negatives': 277}

In [5]:
PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "plain" / "validation.jsonl"
RESULTS_PLAIN_DIR = PROJECT_ROOT / "results" / "plain"

gold_dataset = load_jsonl(DATA_PATH)
labels = get_dataset_labels(gold_dataset)
print(f"Documentos cargados: {len(gold_dataset)}")
print(f"Etiquetas detectadas ({len(labels)}): {labels}")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\riset\\Personal\\Proyectos\\DAM2\\InteligenciaArtificial\\IA\\data\\plain\\validation.jsonl'

In [6]:
import pandas as pd
from ner import evaluate_ner_relaxed

PROJECT_ROOT = Path("..").resolve()
gold_val = load_jsonl(PROJECT_ROOT / "data" / "plain" / "validation.jsonl")
gold_small = load_jsonl(PROJECT_ROOT / "data" / "plain" / "validation_small.jsonl")
labels = get_dataset_labels(gold_val)

def summarize_metrics(name: str, gold, preds) -> pd.DataFrame:
    strict = evaluate_ner_strict(gold, preds)
    relaxed = evaluate_ner_relaxed(gold, preds)

    s_micro = strict.get("micro", strict)
    r_occ = relaxed.get("occurrences", relaxed).get("micro", relaxed.get("occurrences", relaxed))
    r_uniq = relaxed.get("unique", relaxed).get("micro", relaxed.get("unique", relaxed))

    rows = [
        {"Experiment": name, "Eval": "Strict", "P": s_micro.get("precision"), "R": s_micro.get("recall"), "F1": s_micro.get("f1")},
        {"Experiment": name, "Eval": "Relaxed occurrences", "P": r_occ.get("precision"), "R": r_occ.get("recall"), "F1": r_occ.get("f1")},
        {"Experiment": name, "Eval": "Relaxed unique", "P": r_uniq.get("precision"), "R": r_uniq.get("recall"), "F1": r_uniq.get("f1")},
    ]
    return pd.DataFrame(rows)

In [7]:
model_small = load_gliner_model("gliner-community/gliner_small-v2.5")

label_descriptions = {
    lbl: f"Specific {lbl.lower().replace('_', ' ')} mentioned in an official administrative document"
    for lbl in labels
}

preds_small_desc = predict_dataset(model_small, gold_val, labels=label_descriptions, threshold=0.5)
save_responses_to_jsonl(
    preds_small_desc,
    PROJECT_ROOT / "results" / "plain" / "gliner_small_descriptions_validation.jsonl"
)

df_small_desc = summarize_metrics("Plain | gliner_small + desc", gold_val, preds_small_desc)
display(df_small_desc)

C:\Users\riset\Personal\Proyectos\DAM2\InteligenciaArtificial\IA\.venv\Lib\site-packages\huggingface_hub\utils\_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Running GLiNER inference: 0it [00:00, ?it/s]


TypeError: argument should be a str or an os.PathLike object where __fspath__ returns a str, not 'list'

In [8]:
model_large = load_gliner_model("gliner-community/gliner_large-v2.5")

preds_large = predict_dataset(model_large, gold_val, labels=labels, threshold=0.5)
save_responses_to_jsonl(
    preds_large,
    PROJECT_ROOT / "results" / "plain" / "gliner_large_validation.jsonl"
)

df_large = summarize_metrics("Plain | gliner_large", gold_val, preds_large)
display(df_large)

strict_large_full = evaluate_ner_strict(gold_val, preds_large)
print("Desglose por entidad (gliner_large Strict):")
print(strict_large_full)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

C:\Users\riset\Personal\Proyectos\DAM2\InteligenciaArtificial\IA\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\riset\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\riset\Personal\Proyectos\DAM2\InteligenciaArtificial\IA\.venv\Lib\site-packages\huggingface_hub\fi

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

C:\Users\riset\Personal\Proyectos\DAM2\InteligenciaArtificial\IA\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\riset\.cache\huggingface\hub\models--microsoft--deberta-v3-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Running GLiNER inference: 0it [00:00, ?it/s]


TypeError: argument should be a str or an os.PathLike object where __fspath__ returns a str, not 'list'